# ADME public data preparation

---

We used a collection of 3,521 diverse compounds selected from commercially available compound libraries (i.e. Enamine, eMolecules, WuXi LabNetwork, Mcule) and tested against six ADME in vitro assays: HLM, RLM, Solubility, MDR1-MDCK ER, hPPB, and rPPB.

Some basic preprocessing steps, such as data transformations, were applied, data distributions were examined, and a scaffold split is performed.

Source: https://github.com/molecularinformatics/Computational-ADME.git - Dataset to download: 'ADME_public_set_3521.csv'

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
from rdkit import Chem
from chemprop.data.splitting import make_split_indices
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')

In [ ]:
# Load data
inputFile = "../data/ADME_public_set_3521.csv"
df = pd.read_csv(inputFile)
df.head()

In [ ]:
df.shape

In [ ]:
# Remove disconnected SMILES
ixx_keep = ['.' not in x for x in df['SMILES']]
df_filtered = df.iloc[ixx_keep]
df_filtered.shape

In [ ]:
# Convert logPPB to logfu
df_filtered.loc[:,'LOG Fu (HUMAN)'] = [np.nan if np.isnan(x) else x - 2 for x in df_filtered['LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)']]
df_filtered.loc[:,'LOG Fu (RAT)'] = [np.nan if np.isnan(x) else x - 2 for x in df_filtered['LOG PLASMA PROTEIN BINDING (RAT) (% unbound)']]
df_filtered.head()

## Data distributions

In [ ]:
assays_summary = pd.DataFrame()
for c in df_filtered.columns[-8:]:
    assays_summary = assays_summary.append(df_filtered[c].describe())
assays_summary = assays_summary.sort_values('count', ascending=False)
assays_summary

In [ ]:
for c in df_filtered.columns[-8:]:
    plt.figure(figsize=(4, 3))
    sns.histplot(data=df_filtered[df_filtered[c].notna()], x=c)

In [ ]:
df_filtered.shape

## Final dataset

In [ ]:
# Rename endpoints and select columns for final dataset
df_filtered = df_filtered.rename(columns={'Internal ID': 'Id',
                                           'SMILES': 'Structure',
                                           'LOG RLM_CLint (mL/min/kg)': 'rLM LogCLint', 'LOG HLM_CLint (mL/min/kg)': 'hLM LogCLint',
                                           'LOG MDR1-MDCK ER (B-A/A-B)':'MDCK-MDR1_LogER',
                                           'LOG Fu (RAT)': 'LogFu-Rat', 'LOG Fu (HUMAN)': 'LogFu-Human'})

In [ ]:
keep_cols = ['Id', 'Structure', 'rLM LogCLint', 'hLM LogCLint', 'MDCK-MDR1_LogER', 'LogFu-Rat','LogFu-Human']
df_filtered = df_filtered.loc[:, keep_cols]
df_filtered = df_filtered.reset_index(drop=True)
df_filtered.head()

## Save file

Timestamp for saved files

In [ ]:
from datetime import datetime
timestamp_file = datetime.now().strftime("%Y%m%d")
timestamp_file

In [ ]:
outputFileName = f'{inputFile.split(".csv")[0]}_preprocessed{timestamp_file}.csv'
outputFileName

In [ ]:
df_filtered.to_csv(outputFileName, index=False)

## Scaffold-based split

In [ ]:
# Generate RDKit molecule objects
df_filtered['mol'] = df_filtered['Structure'].apply(lambda x: Chem.MolFromSmiles(x))

In [ ]:
# Define the endpoints dictionary
endpoints_dict = {'CLint': ['rLM LogCLint', 'hLM LogCLint'],
                  'MDR1': ['MDCK-MDR1_LogER'],
                  'PPB': ['LogFu-Rat', 'LogFu-Human']}

for endpoint, models_list in endpoints_dict.items():
    print(f'\n### {endpoint} ###\n')

    # Select endpoint data
    df_endpoint = df_filtered[['Id', 'Structure', 'mol'] + models_list]
    df_endpoint = df_endpoint.dropna(subset=models_list, how='all').reset_index(drop=True)
    print(df_endpoint.shape)

    # Split data into training, calibration and test sets (scaffold-based)
    train_idxs, cal_idxs, val_idxs = make_split_indices(df_endpoint['mol'], split='SCAFFOLD_BALANCED', sizes=(0.5, 0.3, 0.2))
    train_idxs, cal_idxs, val_idxs = list(train_idxs[0]), list(cal_idxs[0]), list(val_idxs[0])
    df_endpoint['Subset'] = ''
    df_endpoint.loc[train_idxs, 'Subset'] = 'Training'
    df_endpoint.loc[cal_idxs, 'Subset'] = 'Calibration'
    df_endpoint.loc[val_idxs, 'Subset'] = 'Validation'

    # Select training data and further split it into training and validation
    # sets for model training and early stopping (scaffold-based)
    training_data = df_endpoint.loc[df_endpoint['Subset'] == 'Training'].reset_index(drop=True)
    test_data = df_endpoint.loc[df_endpoint['Subset'] != 'Training'].reset_index(drop=True)
    train_idxs, _, val_idxs = make_split_indices(training_data['mol'], split='SCAFFOLD_BALANCED', sizes=(0.9, 0, 0.1))
    train_idxs, val_idxs = list(train_idxs[0]), list(val_idxs[0])
    training_data['split'] = ''
    training_data.loc[train_idxs, 'split'] = 'train'
    training_data.loc[val_idxs, 'split'] = 'val'
    test_data['split'] = 'test'
    df_endpoint = pd.concat([training_data, test_data], ignore_index=True)
    summary_time_split = pd.DataFrame([df_endpoint[f'Subset'].value_counts()[['Training','Calibration','Validation']], (df_endpoint[f'Subset'].value_counts()/df_endpoint.shape[0]*100)[['Training','Calibration','Validation']]], index=['# Cpds','% Cpds'])
    print(summary_time_split.round())
    summary_time_split = pd.DataFrame([df_endpoint[f'split'].value_counts()[['train','val','test']], (df_endpoint[f'split'].value_counts()/df_endpoint.shape[0]*100)[['train','val','test']]], index=['# Cpds','% Cpds'])
    print(summary_time_split.round())

    # Save preprocessed endpoint dataset with data subsets and splits
    df_endpoint.to_csv(f'./data/{endpoint}_dataset_with_splits.csv')